In [5]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("cwd:", os.getcwd())
print("project root:", PROJECT_ROOT)

cwd: c:\Users\kimdo\Desktop\enko-transformer-lora\notebooks
project root: c:\Users\kimdo\Desktop\enko-transformer-lora


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

from src.model.attention import MultiHeadedAttention

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 2
seq_len = 5
d_model = 512
h = 8

x = torch.randn(batch_size, seq_len, d_model).to(device)
target = torch.randn(batch_size, seq_len, d_model).to(device)

attn = MultiHeadedAttention(
    h=h,
    d_model=d_model,
    dropout=0.1,
    use_lora=True,
    lora_rank=8,
    lora_alpha=16,
    lora_targets=("q", "v")
).to(device)

print("Trainable parameters:")
for name, param in attn.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, attn.parameters()),
    lr=1e-3
)

loss_fn = nn.MSELoss()

for step in range(10):
    optimizer.zero_grad()

    output = attn(x, x, x)
    loss = loss_fn(output, target)

    loss.backward()
    optimizer.step()

    print(f"step {step + 1} | loss: {loss.item():.4f}")

print("output shape:", output.shape)

Trainable parameters:
linears.0.lora_a.weight torch.Size([8, 512])
linears.0.lora_b.weight torch.Size([512, 8])
linears.1.weight torch.Size([512, 512])
linears.1.bias torch.Size([512])
linears.2.lora_a.weight torch.Size([8, 512])
linears.2.lora_b.weight torch.Size([512, 8])
linears.3.weight torch.Size([512, 512])
linears.3.bias torch.Size([512])
step 1 | loss: 1.1035
step 2 | loss: 0.9831
step 3 | loss: 0.8904
step 4 | loss: 0.8028
step 5 | loss: 0.7418
step 6 | loss: 0.6255
step 7 | loss: 0.5403
step 8 | loss: 0.4717
step 9 | loss: 0.3781
step 10 | loss: 0.2992
output shape: torch.Size([2, 5, 512])
